In [ ]:
import pandas as pd
import numpy as np
import os   

In [ ]:
train_df = pd.read_csv("path to file/train_data.csv")
test_df = pd.read_csv("path to file/test_data.csv")

In [ ]:
train_df['folder_name'].value_counts()

In [ ]:
train_image_list = list(train_df['image_path'])
test_image_list = list(test_df['image_path'])
root_dir = "image directory"

In [ ]:
"""
Corrected Angle Extraction
===========================
Drop-in replacement for the angle extraction loop in keypoint_extraction.ipynb.
 
Key fixes:
  1. shoulder_angle  — changed to arm-elevation-from-vertical (virtual point below shoulder)
  2. Variable name collision — rotation angle now stored as 'body_angle' only, never
     overwrites shoulder_angle
  3. front_facing_boolean threshold loosened from 10° to 30°
  4. Added right-side averaging for elbow, hip, knee (uses mean of left+right when both visible)
  5. Added per-landmark visibility check before computing each angle
"""
 
import cv2
import numpy as np
import mediapipe as mp
 
mp_pose = mp.solutions.pose
 
 
# =============================================================================
# ANGLE UTILITY  (unchanged from your original — keep as-is)
# =============================================================================
 
def calculate_angle(a, b, c):
    """
    Angle at point B formed by vectors BA and BC.
    Returns value in [0, 180] degrees.
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - \
              np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle
 
 
# =============================================================================
# NEW: SHOULDER ELEVATION ANGLE  (Fix #1 + Fix #2)
# =============================================================================
 
def calculate_shoulder_elevation(shoulder, elbow):
    """
    Angle of the upper arm relative to a vertical downward reference.
 
    Uses a virtual point directly BELOW the shoulder (same x, y + offset)
    as the reference direction, so the angle is:
        0°   = arm hanging straight down
        90°  = arm horizontal (parallel to ground)
        180° = arm fully raised overhead
 
    This matches the geometric assumption in all cited literature
    (Youdas et al. 2010, Kolber et al. 2014, Fees et al. 1998).
 
    Args:
        shoulder : [x, y] in normalized image coordinates
        elbow    : [x, y] in normalized image coordinates
 
    Returns:
        float: elevation angle in degrees
    """
    # In MediaPipe normalized coords, y increases DOWNWARD.
    # So "directly below" = same x, larger y.
    # Adding 0.1 in normalized space is always safely within frame.
    virtual_below = [shoulder[0], shoulder[1] + 0.1]
    return calculate_angle(virtual_below, shoulder, elbow)
 
 
# =============================================================================
# HELPER: safe landmark getter with visibility check
# =============================================================================
 
def get_lm(landmarks, landmark_enum, min_visibility=0.5):
    """
    Return [x, y] for a landmark if visible, else None.
    Prevents angle calculation on occluded/unreliable joints.
    """
    lm = landmarks[landmark_enum.value]
    if lm.visibility < min_visibility:
        return None
    return [lm.x, lm.y]
 
 
def get_lm_xyz(landmarks, landmark_enum, min_visibility=0.5):
    """Return [x, y, z] for rotation angle calculation."""
    lm = landmarks[landmark_enum.value]
    if lm.visibility < min_visibility:
        return None
    return [lm.x, lm.y, lm.z]
 
 
# =============================================================================
# CORRECTED EXTRACTION LOOP
# =============================================================================
 
shoulder_angles      = []   # arm elevation from vertical (FIX #1)
elbow_angles         = []   # elbow flexion — unchanged, was correct
hip_angles           = []   # hip flexion — unchanged, was correct
knee_angles          = []   # knee flexion — unchanged, was correct
body_angles          = []   # body rotation (was wrongly named shoulder_angle before — FIX #2)
front_facing_boolean = []
 
PL = mp_pose.PoseLandmark  # shorthand
 
def process_images(image_list):
    for i, image in enumerate(image_list):
        image_file = root_dir + image
    
        with mp_pose.Pose(
            static_image_mode=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as pose:
    
            image_cv = cv2.imread(image_file)
            if image_cv is None:
                print(f'Image {i} could not be read: {image_file}')
                for lst in [shoulder_angles, elbow_angles, hip_angles,
                            knee_angles, body_angles, front_facing_boolean]:
                    lst.append("Not Found")
                continue
    
            results = pose.process(cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB))
    
            try:
                landmarks = results.pose_landmarks.landmark
    
                # ------------------------------------------------------------------
                # 1. SHOULDER ELEVATION ANGLE  (FIX #1: virtual-below reference)
                #    Uses LEFT side; falls back to RIGHT if left not visible.
                # ------------------------------------------------------------------
                l_shoulder = get_lm(landmarks, PL.LEFT_SHOULDER)
                l_elbow    = get_lm(landmarks, PL.LEFT_ELBOW)
                r_shoulder = get_lm(landmarks, PL.RIGHT_SHOULDER)
                r_elbow    = get_lm(landmarks, PL.RIGHT_ELBOW)
    
                if l_shoulder and l_elbow:
                    s_angle = calculate_shoulder_elevation(l_shoulder, l_elbow)
                elif r_shoulder and r_elbow:
                    # Mirror x so the virtual-below reference stays correct
                    r_shoulder_m = [1 - r_shoulder[0], r_shoulder[1]]
                    r_elbow_m    = [1 - r_elbow[0],    r_elbow[1]]
                    s_angle = calculate_shoulder_elevation(r_shoulder_m, r_elbow_m)
                else:
                    s_angle = "Not Found"
    
                shoulder_angles.append(s_angle)
    
                # ------------------------------------------------------------------
                # 2. ELBOW ANGLE  (unchanged — was correct)
                #    shoulder → elbow → wrist
                # ------------------------------------------------------------------
                shoulder_pt = get_lm(landmarks, PL.LEFT_SHOULDER)
                elbow_pt    = get_lm(landmarks, PL.LEFT_ELBOW)
                wrist_pt    = get_lm(landmarks, PL.LEFT_WRIST)
    
                if shoulder_pt and elbow_pt and wrist_pt:
                    e_angle = calculate_angle(shoulder_pt, elbow_pt, wrist_pt)
                else:
                    # Try right side
                    rs = get_lm(landmarks, PL.RIGHT_SHOULDER)
                    re = get_lm(landmarks, PL.RIGHT_ELBOW)
                    rw = get_lm(landmarks, PL.RIGHT_WRIST)
                    e_angle = calculate_angle(rs, re, rw) if (rs and re and rw) else "Not Found"
    
                elbow_angles.append(e_angle)
    
                # ------------------------------------------------------------------
                # 3. HIP ANGLE  (unchanged — was correct)
                #    shoulder → hip → knee
                # ------------------------------------------------------------------
                shoulder_pt = get_lm(landmarks, PL.LEFT_SHOULDER)
                hip_pt      = get_lm(landmarks, PL.LEFT_HIP)
                knee_pt     = get_lm(landmarks, PL.LEFT_KNEE)
    
                if shoulder_pt and hip_pt and knee_pt:
                    h_angle = calculate_angle(shoulder_pt, hip_pt, knee_pt)
                else:
                    rs = get_lm(landmarks, PL.RIGHT_SHOULDER)
                    rh = get_lm(landmarks, PL.RIGHT_HIP)
                    rk = get_lm(landmarks, PL.RIGHT_KNEE)
                    h_angle = calculate_angle(rs, rh, rk) if (rs and rh and rk) else "Not Found"
    
                hip_angles.append(h_angle)
    
                # ------------------------------------------------------------------
                # 4. KNEE ANGLE  (unchanged — was correct)
                #    hip → knee → ankle
                # ------------------------------------------------------------------
                hip_pt   = get_lm(landmarks, PL.LEFT_HIP)
                knee_pt  = get_lm(landmarks, PL.LEFT_KNEE)
                ankle_pt = get_lm(landmarks, PL.LEFT_ANKLE)
    
                if hip_pt and knee_pt and ankle_pt:
                    k_angle = calculate_angle(hip_pt, knee_pt, ankle_pt)
                else:
                    rh = get_lm(landmarks, PL.RIGHT_HIP)
                    rk = get_lm(landmarks, PL.RIGHT_KNEE)
                    ra = get_lm(landmarks, PL.RIGHT_ANKLE)
                    k_angle = calculate_angle(rh, rk, ra) if (rh and rk and ra) else "Not Found"
    
                knee_angles.append(k_angle)
    
                # ------------------------------------------------------------------
                # 5. BODY ROTATION ANGLE  (FIX #2: renamed, no longer overwrites shoulder_angle)
                #    Uses z-depth difference between left/right shoulders and hips.
                # ------------------------------------------------------------------
                ls_xyz = get_lm_xyz(landmarks, PL.LEFT_SHOULDER,  min_visibility=0.3)
                rs_xyz = get_lm_xyz(landmarks, PL.RIGHT_SHOULDER, min_visibility=0.3)
                lh_xyz = get_lm_xyz(landmarks, PL.LEFT_HIP,       min_visibility=0.3)
                rh_xyz = get_lm_xyz(landmarks, PL.RIGHT_HIP,      min_visibility=0.3)
    
                rotation_parts = []
                if ls_xyz and rs_xyz:
                    shoulder_rot = np.arctan2(
                        rs_xyz[2] - ls_xyz[2],
                        rs_xyz[0] - ls_xyz[0]
                    ) * (180 / np.pi)
                    rotation_parts.append(shoulder_rot)
    
                if lh_xyz and rh_xyz:
                    hip_rot = np.arctan2(
                        rh_xyz[2] - lh_xyz[2],
                        rh_xyz[0] - lh_xyz[0]
                    ) * (180 / np.pi)
                    rotation_parts.append(hip_rot)
    
                if rotation_parts:
                    rotation_angle = np.mean(rotation_parts)  # average shoulder + hip rotation
                    body_angles.append(rotation_angle)
                    # FIX #3: loosened threshold from 10° to 30°
                    front_facing_boolean.append(abs(rotation_angle) < 30)
                else:
                    body_angles.append("Not Found")
                    front_facing_boolean.append("Not Found")
    
                print(f'Image {i} processed successfully')
    
            except Exception as e:
                print(f'Image {i} failed with error: {e}')
                for lst in [shoulder_angles, elbow_angles, hip_angles,
                            knee_angles, body_angles, front_facing_boolean]:
                    lst.append("Not Found")
    return shoulder_angles, elbow_angles, hip_angles, knee_angles, body_angles, front_facing_boolean    


In [ ]:
shoulder_angles, elbow_angles, hip_angles, knee_angles, body_angles, front_facing_boolean = process_images(train_image_list)    

In [ ]:
train_df['shoulder_angles'] = shoulder_angles
train_df['elbow_angles'] = elbow_angles
train_df['hip_angles'] = hip_angles
train_df['knee_angles'] = knee_angles
train_df['front_facing_boolean'] = front_facing_boolean
train_df['body_angles'] = body_angles

In [ ]:
shoulder_angles, elbow_angles, hip_angles, knee_angles, body_angles, front_facing_boolean = process_images(test_image_list)  

In [ ]:
test_df['shoulder_angles'] = shoulder_angles
test_df['elbow_angles'] = elbow_angles
test_df['hip_angles'] = hip_angles
test_df['knee_angles'] = knee_angles
test_df['front_facing_boolean'] = front_facing_boolean
test_df['body_angles'] = body_angles

In [ ]:
train_df.to_csv("train_data_with_extracted_features.csv", index=False)
test_df.to_csv("test_data_with_extracted_features.csv", index=False)